# ai-learn-19: agent memory

Replay one seeded conversation through each memory strategy and look at what each one would put in the prompt when asked about an old fact.

In [ ]:
import sys; sys.path.insert(0, '..')
from conversation import make_episodes
from memory import FullHistory, WindowBuffer, VectorMemory, RollingSummary, HybridMemory, extractive_reader, n_tokens
from evaluate import mentions
turns, probes = make_episodes(1, 80, 42)[0]
for t in turns[:6]:
    print(t.idx, 'USER:', t.user, '| AGENT:', t.agent)
print('facts at turns:', sorted((p.fact_turn, p.slot) for p in probes))

## 1. Feed the conversation to every memory

In [ ]:
mems = [FullHistory(), WindowBuffer(6), VectorMemory(3), RollingSummary(8, 2, 60), HybridMemory(4, 2, 8, 2, 40)]
for t in turns:
    for m in mems:
        m.observe(t)
oldest = min(probes, key=lambda p: p.fact_turn)
print('question:', oldest.question, '| gold:', oldest.answer, '| stated', 80 - oldest.fact_turn, 'turns ago')

## 2. What each memory returns, and what the toy reader picks

In [ ]:
for m in mems:
    ctx = m.context(oldest.question)
    pick = extractive_reader(oldest.question, ctx)
    print(f'--- {m.name}: {len(ctx)} lines, {n_tokens(ctx)} words, fact in context={any(mentions(l, oldest.answer) for l in ctx)}')
    print('    reader picked:', pick)
print()
print('rolling summary lines:', mems[3].summary)

## 3. Full smoke run (40 conversations)
Regenerates `results/`.

In [ ]:
import os
os.chdir('..')
import run_smoke
m = run_smoke.main()
for s, r in m['results'].items():
    print(f"{s:13s} contain={r['contain']:.3f} reader={r['reader_acc']:.3f} words@probe={r['ctx_tokens_mean']:.1f}")